In [2]:
import pandas as pd
import urllib.request
from sklearn.model_selection import train_test_split
from surprise import Dataset, Reader, SVD

# 1. ЗАГРУЗКА ДАННЫХ
print("1. Скачиваем данные...")
url = "https://raw.githubusercontent.com/Julia-Koval/Amazon-Reviews-Dataset/master/Video_Games_5.json"
filename = "amazon_data.json"
urllib.request.urlretrieve(url, filename)
df = pd.read_json(filename, lines=True)

# 2. ЧИСТКА
print("2. Чистим данные...")
df_clean = df.drop_duplicates(subset=['reviewerID', 'asin'])
df_clean = df_clean[['reviewerID', 'asin', 'overall', 'reviewText', 'summary']]

# 3. РАЗБИЕНИЕ
print("3. Делим на train/test...")
train_df, test_df = train_test_split(df_clean, test_size=0.2, random_state=42)
test_users = test_df['reviewerID'].unique()[:100]

# 4. МЕТРИКА
def precision_at_k(recommended_items, actual_items, k=10):
    top_k = recommended_items[:k]
    hits = len(set(top_k) & set(actual_items))
    return hits / k

# 5. BASELINE
print("4. Считаем Baseline...")
item_stats = df_clean.groupby('asin').agg(avg_rating=('overall', 'mean'), review_count=('overall', 'count')).reset_index()
item_stats = item_stats[item_stats['review_count'] >= 10]
top_popular_items = item_stats.sort_values(by=['avg_rating', 'review_count'], ascending=[False, False])
baseline_recommendations = top_popular_items.head(10)['asin'].tolist()

# 6. SVD
print("5. Обучаем SVD...")
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(train_df[['reviewerID', 'asin', 'overall']], reader)
trainset = data.build_full_trainset()
algo = SVD(n_factors=50, random_state=42)
algo.fit(trainset)

print("6. Считаем точность SVD...")
def get_svd_recommendations(user_id, algo, all_items, n=10):
    predictions = []
    for item in all_items:
        pred = algo.predict(user_id, item)
        predictions.append((item, pred.est))
    predictions.sort(key=lambda x: x[1], reverse=True)
    return [x[0] for x in predictions[:n]]

all_unique_items = df_clean['asin'].unique().tolist()

svd_precisions = []
for user in test_users:
    actual_items = test_df[test_df['reviewerID'] == user]['asin'].tolist()
    svd_recs = get_svd_recommendations(user, algo, all_unique_items, n=10)
    svd_precisions.append(precision_at_k(svd_recs, actual_items, k=10))

avg_svd_precision = sum(svd_precisions) / len(svd_precisions)
print(f"=== СРЕДНИЙ PRECISION@10 ДЛЯ SVD: {avg_svd_precision:.4f} ===")

1. Скачиваем данные...


HTTPError: HTTP Error 404: Not Found

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from surprise import Dataset, Reader, SVD
import os

# 1. ЗАГРУЗКА ДАННЫХ (Локальный файл)
# Проверяем, какой файл есть в папке
if os.path.exists("Video_Games_5.json"):
    filename = "Video_Games_5.json"
elif os.path.exists("amazon_data.json"):
    filename = "amazon_data.json"
else:
    print("ФАЙЛ НЕ НАЙДЕН! Скачай его заново в папку проекта.")
    raise FileNotFoundError

print(f"1. Читаем данные из {filename}...")
df = pd.read_json(filename, lines=True)

# 2. ЧИСТКА
print("2. Чистим данные...")
df_clean = df.drop_duplicates(subset=['reviewerID', 'asin'])
df_clean = df_clean[['reviewerID', 'asin', 'overall', 'reviewText', 'summary']]

# 3. РАЗБИЕНИЕ
print("3. Делим на train/test...")
train_df, test_df = train_test_split(df_clean, test_size=0.2, random_state=42)
test_users = test_df['reviewerID'].unique()[:100]

# 4. МЕТРИКА
def precision_at_k(recommended_items, actual_items, k=10):
    top_k = recommended_items[:k]
    hits = len(set(top_k) & set(actual_items))
    return hits / k

# 5. BASELINE
print("4. Считаем Baseline...")
item_stats = df_clean.groupby('asin').agg(avg_rating=('overall', 'mean'), review_count=('overall', 'count')).reset_index()
item_stats = item_stats[item_stats['review_count'] >= 10]
top_popular_items = item_stats.sort_values(by=['avg_rating', 'review_count'], ascending=[False, False])
baseline_recommendations = top_popular_items.head(10)['asin'].tolist()

# 6. SVD
print("5. Обучаем SVD...")
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(train_df[['reviewerID', 'asin', 'overall']], reader)
trainset = data.build_full_trainset()
algo = SVD(n_factors=50, random_state=42)
algo.fit(trainset)

print("6. Считаем точность SVD...")
def get_svd_recommendations(user_id, algo, all_items, n=10):
    predictions = []
    for item in all_items:
        pred = algo.predict(user_id, item)
        predictions.append((item, pred.est))
    predictions.sort(key=lambda x: x[1], reverse=True)
    return [x[0] for x in predictions[:n]]

all_unique_items = df_clean['asin'].unique().tolist()

svd_precisions = []
for user in test_users:
    actual_items = test_df[test_df['reviewerID'] == user]['asin'].tolist()
    svd_recs = get_svd_recommendations(user, algo, all_unique_items, n=10)
    svd_precisions.append(precision_at_k(svd_recs, actual_items, k=10))

avg_svd_precision = sum(svd_precisions) / len(svd_precisions)
print(f"=== СРЕДНИЙ PRECISION@10 ДЛЯ SVD: {avg_svd_precision:.4f} ===")

ФАЙЛ НЕ НАЙДЕН! Скачай его заново в папку проекта.


FileNotFoundError: 

In [4]:
import pandas as pd
import urllib.request
from sklearn.model_selection import train_test_split
from surprise import Dataset, Reader, SVD

# 1. ЗАГРУЗКА ДАННЫХ (Стабильная ссылка UCSD)
print("1. Скачиваем данные с официального сервера...")
url = "https://datarepo.eng.ucsd.edu/mcauley_group/data/amazon_v2/categoryFilesSmall/Video_Games_5.json.gz"
filename = "Video_Games_5.json.gz"

# Скачиваем сжатый файл
urllib.request.urlretrieve(url, filename)
print("Данные скачаны! Читаем...")
# Читаем напрямую из сжатого архива (compression='gzip')
df = pd.read_json(filename, lines=True, compression='gzip')

# 2. ЧИСТКА
print("2. Чистим данные...")
df_clean = df.drop_duplicates(subset=['reviewerID', 'asin'])
df_clean = df_clean[['reviewerID', 'asin', 'overall', 'reviewText', 'summary']]

# 3. РАЗБИЕНИЕ
print("3. Делим на train/test...")
train_df, test_df = train_test_split(df_clean, test_size=0.2, random_state=42)
test_users = test_df['reviewerID'].unique()[:100]

# 4. МЕТРИКА
def precision_at_k(recommended_items, actual_items, k=10):
    top_k = recommended_items[:k]
    hits = len(set(top_k) & set(actual_items))
    return hits / k

# 5. BASELINE
print("4. Считаем Baseline...")
item_stats = df_clean.groupby('asin').agg(avg_rating=('overall', 'mean'), review_count=('overall', 'count')).reset_index()
item_stats = item_stats[item_stats['review_count'] >= 10]
top_popular_items = item_stats.sort_values(by=['avg_rating', 'review_count'], ascending=[False, False])
baseline_recommendations = top_popular_items.head(10)['asin'].tolist()

# 6. SVD
print("5. Обучаем SVD...")
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(train_df[['reviewerID', 'asin', 'overall']], reader)
trainset = data.build_full_trainset()
algo = SVD(n_factors=50, random_state=42)
algo.fit(trainset)

print("6. Считаем точность SVD...")
def get_svd_recommendations(user_id, algo, all_items, n=10):
    predictions = []
    for item in all_items:
        pred = algo.predict(user_id, item)
        predictions.append((item, pred.est))
    predictions.sort(key=lambda x: x[1], reverse=True)
    return [x[0] for x in predictions[:n]]

all_unique_items = df_clean['asin'].unique().tolist()

svd_precisions = []
for user in test_users:
    actual_items = test_df[test_df['reviewerID'] == user]['asin'].tolist()
    svd_recs = get_svd_recommendations(user, algo, all_unique_items, n=10)
    svd_precisions.append(precision_at_k(svd_recs, actual_items, k=10))

avg_svd_precision = sum(svd_precisions) / len(svd_precisions)
print(f"=== СРЕДНИЙ PRECISION@10 ДЛЯ SVD: {avg_svd_precision:.4f} ===")

1. Скачиваем данные с официального сервера...


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)>

In [5]:
import pandas as pd
import urllib.request
import gzip
import shutil
from sklearn.model_selection import train_test_split
from surprise import Dataset, Reader, SVD

# 1. ЗАГРУЗКА ДАННЫХ (Ссылка Стэнфорда)
print("1. Скачиваем данные...")
url = "https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Toys_and_Games_5.json.gz"
gz_filename = "reviews_Toys_and_Games_5.json.gz"
json_filename = "reviews_Toys_and_Games_5.json"

# Скачиваем
urllib.request.urlretrieve(url, gz_filename)
print("Файл скачан! Распаковываем...")

# Распаковываем .gz в обычный .json
with gzip.open(gz_filename, 'rb') as f_in:
    with open(json_filename, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

print("Читаем данные в таблицу...")
df = pd.read_json(json_filename, lines=True)

# 2. ЧИСТКА
print("2. Чистим данные...")
df_clean = df.drop_duplicates(subset=['reviewerID', 'asin'])
# Проверяем, что нужные колонки вообще есть
cols = ['reviewerID', 'asin', 'overall', 'reviewText', 'summary']
df_clean = df_clean[[c for c in cols if c in df_clean.columns]]

# 3. РАЗБИЕНИЕ
print("3. Делим на train/test...")
train_df, test_df = train_test_split(df_clean, test_size=0.2, random_state=42)
test_users = test_df['reviewerID'].unique()[:100]

# 4. МЕТРИКА
def precision_at_k(recommended_items, actual_items, k=10):
    top_k = recommended_items[:k]
    hits = len(set(top_k) & set(actual_items))
    return hits / k

# 5. BASELINE
print("4. Считаем Baseline...")
item_stats = df_clean.groupby('asin').agg(avg_rating=('overall', 'mean'), review_count=('overall', 'count')).reset_index()
item_stats = item_stats[item_stats['review_count'] >= 10]
top_popular_items = item_stats.sort_values(by=['avg_rating', 'review_count'], ascending=[False, False])
baseline_recommendations = top_popular_items.head(10)['asin'].tolist()

# 6. SVD
print("5. Обучаем SVD...")
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(train_df[['reviewerID', 'asin', 'overall']], reader)
trainset = data.build_full_trainset()
algo = SVD(n_factors=50, random_state=42)
algo.fit(trainset)

print("6. Считаем точность SVD...")
def get_svd_recommendations(user_id, algo, all_items, n=10):
    predictions = []
    for item in all_items:
        pred = algo.predict(user_id, item)
        predictions.append((item, pred.est))
    predictions.sort(key=lambda x: x[1], reverse=True)
    return [x[0] for x in predictions[:n]]

all_unique_items = df_clean['asin'].unique().tolist()

svd_precisions = []
for user in test_users:
    actual_items = test_df[test_df['reviewerID'] == user]['asin'].tolist()
    svd_recs = get_svd_recommendations(user, algo, all_unique_items, n=10)
    svd_precisions.append(precision_at_k(svd_recs, actual_items, k=10))

avg_svd_precision = sum(svd_precisions) / len(svd_precisions)
print(f"=== СРЕДНИЙ PRECISION@10 ДЛЯ SVD: {avg_svd_precision:.4f} ===")

1. Скачиваем данные...
Файл скачан! Распаковываем...
Читаем данные в таблицу...
2. Чистим данные...
3. Делим на train/test...
4. Считаем Baseline...
5. Обучаем SVD...
6. Считаем точность SVD...
=== СРЕДНИЙ PRECISION@10 ДЛЯ SVD: 0.0030 ===


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("1. Собираем тексты для каждого товара...")
# Группируем по товару и склеиваем все отзывы в один большой текст
item_texts = df_clean.groupby('asin')['reviewText'].apply(lambda x: ' '.join(x.astype(str))).reset_index()

print("2. Обучаем TF-IDF...")
# Берем максимум 5000 самых важных слов, чтобы не перегружать память
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(item_texts['reviewText'])

# Создаем словарь: ID товара -> его индекс в матрице
item_id_to_index = {item_id: idx for idx, item_id in enumerate(item_texts['asin'])}

print("3. Считаем похожесть товаров...")
# Считаем косинусное сходство между всеми товарами
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Функция для получения рекомендаций на основе текста
def get_content_recommendations(user_id, user_history, item_id_to_index, cosine_sim, all_items, n=10):
    # Находим индексы товаров, которые пользователь уже оценил
    user_indices = [item_id_to_index[item] for item in user_history if item in item_id_to_index]
    
    if not user_indices:
        return [] # Если истории нет, ничего не рекомендуем
    
    # Складываем вектора похожести для всех купленных товаров
    sim_scores = np.zeros(len(item_id_to_index))
    for idx in user_indices:
        sim_scores += cosine_sim[idx]
        
    # Сортируем по убыванию
    sim_scores_indices = np.argsort(sim_scores)[::-1]
    
    # Достаем ID товаров, исключая те, что пользователь уже видел
    recommended = []
    for i in sim_scores_indices:
        item_id = item_texts['asin'].iloc[i]
        if item_id not in user_history:
            recommended.append(item_id)
        if len(recommended) == n:
            break
    return recommended

print("Текстовая модель готова!")

1. Собираем тексты для каждого товара...
2. Обучаем TF-IDF...
3. Считаем похожесть товаров...
Текстовая модель готова!


In [7]:
print("Считаем точность Ансамбля...")
ensemble_precisions = []
alpha = 0.5 # Вес SVD (0.5 SVD, 0.5 TF-IDF). Позже мы его покрутим.

for user in test_users:
    # Что пользователь реально купил в тесте
    actual_items = test_df[test_df['reviewerID'] == user]['asin'].tolist()
    # Что он покупал в трейне (история)
    user_history = train_df[train_df['reviewerID'] == user]['asin'].tolist()
    
    # 1. Получаем рекомендации от SVD
    svd_recs = get_svd_recommendations(user, algo, all_unique_items, n=50) # Берем топ-50 для запаса
    
    # 2. Получаем рекомендации от Текстовой модели
    tfidf_recs = get_content_recommendations(user, user_history, item_id_to_index, cosine_sim, all_unique_items, n=50)
    
    # 3. Смешиваем (Ансамбль)
    # Даем каждому товару баллы: от SVD (по позиции) и от TF-IDF (по позиции)
    scores = {}
    for i, item in enumerate(svd_recs):
        scores[item] = scores.get(item, 0) + alpha * (50 - i) # Чем выше в топе, тем больше баллов
    
    for i, item in enumerate(tfidf_recs):
        scores[item] = scores.get(item, 0) + (1 - alpha) * (50 - i)
        
    # Сортируем по итоговому баллу и берем топ-10
    final_recs = sorted(scores.keys(), key=lambda x: scores[x], reverse=True)[:10]
    
    # Считаем Precision
    ensemble_precisions.append(precision_at_k(final_recs, actual_items, k=10))

avg_ensemble_precision = sum(ensemble_precisions) / len(ensemble_precisions)
print(f"=== СРЕДНИЙ PRECISION@10 ДЛЯ АНСАМБЛЯ: {avg_ensemble_precision:.4f} ===")

Считаем точность Ансамбля...
=== СРЕДНИЙ PRECISION@10 ДЛЯ АНСАМБЛЯ: 0.0100 ===
